In [1]:
import pandas as pd
import nltk

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

import tensorflow as tf

In [2]:
df = pd.read_csv("../Dataset/emails.csv")
df.head()

,subject,message,label,text
0,Re: New Sequences Window,"Date: Wed, 21 Aug 2002 10:54:46 -05...",ham,"Re: New Sequences Window Date: Wed,..."
1,[zzzzteana] RE: Alexander,"Martin A posted:\nTassos Papadopoulos, the Gre...",ham,[zzzzteana] RE: Alexander Martin A posted:\nTa...
2,[zzzzteana] Moscow bomber,Man Threatens Explosion In Moscow \n\nThursday...,ham,[zzzzteana] Moscow bomber Man Threatens Explos...
3,[IRR] Klez: The Virus That Won't Die,Klez: The Virus That Won't Die\n \nAlready the...,ham,[IRR] Klez: The Virus That Won't Die Klez: Th...
4,Re: [zzzzteana] Nothing like mama used to make,"> in adding cream to spaghetti carbonara, whi...",ham,Re: [zzzzteana] Nothing like mama used to make...


In [3]:
df['text'] = df['subject'].fillna("") + " " + df['message'].fillna("")
df.head()

,subject,message,label,text
0,Re: New Sequences Window,"Date: Wed, 21 Aug 2002 10:54:46 -05...",ham,"Re: New Sequences Window Date: Wed,..."
1,[zzzzteana] RE: Alexander,"Martin A posted:\nTassos Papadopoulos, the Gre...",ham,[zzzzteana] RE: Alexander Martin A posted:\nTa...
2,[zzzzteana] Moscow bomber,Man Threatens Explosion In Moscow \n\nThursday...,ham,[zzzzteana] Moscow bomber Man Threatens Explos...
3,[IRR] Klez: The Virus That Won't Die,Klez: The Virus That Won't Die\n \nAlready the...,ham,[IRR] Klez: The Virus That Won't Die Klez: Th...
4,Re: [zzzzteana] Nothing like mama used to make,"> in adding cream to spaghetti carbonara, whi...",ham,Re: [zzzzteana] Nothing like mama used to make...


In [4]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

In [5]:
def preprocess_text(text):
    text = text.lower()
    tokens = word_tokenize(text)
    clean_tokens = []

    for word in tokens:
        if word.isalpha() and word not in stop_words:
            word = lemmatizer.lemmatize(word)
            clean_tokens.append(word)

    return " ".join(clean_tokens)

In [6]:
df["clean_text"] = df["text"].apply(preprocess_text)

df[["text", "clean_text", "label"]].head()

,text,clean_text,label
0,"Re: New Sequences Window Date: Wed,...",new sequence window date wed aug chris garrigu...,ham
1,[zzzzteana] RE: Alexander Martin A posted:\nTa...,zzzzteana alexander martin posted tasso papado...,ham
2,[zzzzteana] Moscow bomber Man Threatens Explos...,zzzzteana moscow bomber man threatens explosio...,ham
3,[IRR] Klez: The Virus That Won't Die Klez: Th...,irr klez virus wo die klez virus wo die alread...,ham
4,Re: [zzzzteana] Nothing like mama used to make...,zzzzteana nothing like mama used make adding c...,ham


In [7]:
vectorizer = TfidfVectorizer(max_features=5000,ngram_range=(1,2),min_df=2)
x=vectorizer.fit_transform(df['clean_text'])


In [8]:
y=df['label'].map({"ham":0,"spam":1})


In [9]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)
x_train = x_train.toarray()
x_test = x_test.toarray()


In [10]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(x_train.shape[1],)),
    tf.keras.layers.Dense(128,activation='relu'),
    tf.keras.layers.Dense(64,activation='relu'),
    tf.keras.layers.Dense(1,activation='sigmoid')
])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │       640,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 648,449 (2.47 MB)

 Trainable params: 648,449 (2.47 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
model.compile(optimizer='adam',loss="binary_crossentropy",metrics=['accuracy'])
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

In [12]:
history = model.fit(
    x_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop]
)

Epoch 1/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 5s 30ms/step - accuracy: 0.8698 - loss: 0.3459 - val_accuracy: 0.9647 - val_loss: 0.1253
Epoch 2/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.9875 - loss: 0.0488 - val_accuracy: 0.9917 - val_loss: 0.0269
Epoch 3/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.9990 - loss: 0.0083 - val_accuracy: 0.9917 - val_loss: 0.0185
Epoch 4/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 1.0000 - loss: 0.0031 - val_accuracy: 0.9917 - val_loss: 0.0201
Epoch 5/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 1.0000 - loss: 0.0017 - val_accuracy: 0.9917 - val_loss: 0.0180
Epoch 6/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy: 1.0000 - loss: 0.0011 - val_accuracy: 0.9896 - val_loss: 0.0175
Epoch 7/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 1.0000 - loss: 7.4413e-04 - val_accuracy: 0.9917 - val_loss: 0.0171
Epoch 8/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 1.0000 - loss: 5.4070e-04 - val_accuracy: 0.

In [ ]:
model.save("ann_model.keras")


In [16]:
import joblib
joblib.dump(vectorizer,"ann_vectorizer.pkl")


['ann_vectorizer.pkl']